In [2]:
!nvidia-smi
!pip install -q -U transformers datasets accelerate

import torch
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))
print("HF user:", whoami()["name"])
print("torch:", torch.__version__, "| GPU var mi:", torch.cuda.is_available())

Thu Nov 12 00:05:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

LocalProtocolError: Illegal header value b'Bearer hf_REDACTED\r\n'

In [3]:
import torch
from google.colab import userdata
from huggingface_hub import login, whoami

token = userdata.get("HF_TOKEN").strip()
print("token uzunlugu:", len(token), "| hf_ ile basliyor mu:", token.startswith("hf_"))
login(token=token)
print("HF user:", whoami()["name"])
print("torch:", torch.__version__, "| GPU var mi:", torch.cuda.is_available())

token uzunlugu: 37 | hf_ ile basliyor mu: True
HF user: goncayldz
torch: 2.14.1+cu130 | GPU var mi: True


In [4]:
import re
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN").strip())

# Mac'teki src/preprocess.py ile AYNI fonksiyon
def normalize(text):
    t = text.lower()
    t = t.replace("\u2019", "'").replace("\u2018", "'")
    t = re.sub(r"[^\w' ]+|_", " ", t)
    t = re.sub(r"([^\W\d_])\1{2,}", r"\1\1", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

MODEL = "bert-base-uncased"
MAX_LEN = 40

ds = load_dataset("clinc/clinc_oos", "plus")
names = ds["train"].features["intent"].names
OOS = names.index("oos")
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(split):
    texts = [normalize(t) for t in ds[split]["text"]]
    lens = [len(x) for x in tok(texts)["input_ids"]]
    print(f"{split}: {len(texts)} rows | longest {max(lens)} tokens | longer than {MAX_LEN}: {sum(l > MAX_LEN for l in lens)}")
    enc = tok(texts, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    y = torch.tensor(ds[split]["intent"])
    return enc["input_ids"], enc["attention_mask"], y

tr_ids, tr_mask, tr_y = encode("train")
va_ids, va_mask, va_y = encode("validation")
te_ids, te_mask, te_y = encode("test")
print("labels:", len(names), "| oos id:", OOS)
print(tok.convert_ids_to_tokens(tr_ids[0])[:12])

README.md:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

plus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  312kB            

plus/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plus/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 77.8kB            

plus/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plus/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  136kB            

... (5 outputs left out)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

train: 15250 rows | longest 32 tokens | longer than 40: 0
validation: 3100 rows | longest 28 tokens | longer than 40: 0
test: 5500 rows | longest 30 tokens | longer than 40: 0
labels: 151 | oos id: 42
['[CLS]', 'what', 'expression', 'would', 'i', 'use', 'to', 'say', 'i', 'love', 'you', 'if']


In [5]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification

torch.manual_seed(0)
dev = "cuda"
EPOCHS, BATCH, LR = 5, 32, 5e-5
REPO = "goncayldz/clinc150-bert-teacher"

# ---- metrikler (Mac'teki src/metrics.py ile aynı mantık) ----
def final_predictions(probs, oos_id, tau):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    return np.where(conf < tau, oos_id, pred)

def in_scope_accuracy(y_true, y_pred, oos_id):
    m = y_true != oos_id
    return float((y_pred[m] == y_true[m]).mean())

def oos_recall(y_true, y_pred, oos_id):
    m = y_true == oos_id
    return float((y_pred[m] == oos_id).mean())

def tune_tau(probs, y_true, oos_id):
    best = None
    for tau in np.arange(0.0, 1.0, 0.02):
        p = final_predictions(probs, oos_id, tau)
        acc = in_scope_accuracy(y_true, p, oos_id)
        rec = oos_recall(y_true, p, oos_id)
        score = (acc + rec) / 2
        if best is None or score > best[0]:
            best = (score, float(tau), acc, rec)
    return best

@torch.no_grad()
def get_probs(model, ids, mask, bs=256):
    model.eval()
    out = []
    for i in range(0, len(ids), bs):
        logits = model(input_ids=ids[i:i + bs].to(dev), attention_mask=mask[i:i + bs].to(dev)).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

# ---- model ----
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(names),
    id2label={i: n for i, n in enumerate(names)},
    label2id={n: i for i, n in enumerate(names)},
).to(dev)

loader = DataLoader(TensorDataset(tr_ids, tr_mask, tr_y), batch_size=BATCH, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * len(loader)
warm = int(0.1 * steps)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / (steps - warm)))
)

yva = va_y.numpy()
best = {"score": -1.0}
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for ids, mask, y in loader:
        ids, mask, y = ids.to(dev), mask.to(dev), y.to(dev)
        opt.zero_grad()
        loss = model(input_ids=ids, attention_mask=mask, labels=y).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        total += loss.item() * len(y)

    probs = get_probs(model, va_ids, va_mask)
    score, tau, acc, rec = tune_tau(probs, yva, OOS)
    argmax_acc = in_scope_accuracy(yva, probs.argmax(axis=1), OOS)
    print(
        f"epoch {epoch} | train loss {total / len(tr_y):.4f} | val score {score:.4f} "
        f"(acc {acc:.4f}, oos recall {rec:.4f}, tau {tau:.2f}) | argmax acc {argmax_acc:.4f} "
        f"[{time.time() - t0:.0f}s]"
    )
    if score > best["score"]:
        best = {"score": score, "tau": tau, "acc": acc, "rec": rec, "epoch": epoch,
                "argmax_acc": argmax_acc,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}

print(f"\nBEST epoch {best['epoch']} | tau {best['tau']:.2f} | in-scope acc {best['acc']:.4f} | "
      f"oos recall {best['rec']:.4f} | score {best['score']:.4f} | argmax acc {best['argmax_acc']:.4f}")

# ---- en iyi epoch'u yükle ve Hub'a gönder ----
model.load_state_dict(best["state"])
model.push_to_hub(REPO)
tok.push_to_hub(REPO)

import json
from huggingface_hub import upload_file
json.dump({"tau": best["tau"], "epoch": best["epoch"]}, open("tau.json", "w"))
upload_file(path_or_fileobj="tau.json", path_in_repo="tau.json", repo_id=REPO)
print("uploaded: https://huggingface.co/" + REPO)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1 | train loss 3.4601 | val score 0.8703 (acc 0.8607, oos recall 0.8800, tau 0.18) | argmax acc 0.9127 [111s]
epoch 2 | train loss 0.5526 | val score 0.9367 (acc 0.9133, oos recall 0.9600, tau 0.70) | argmax acc 0.9583 [233s]
epoch 3 | train loss 0.1095 | val score 0.9435 (acc 0.9370, oos recall 0.9500, tau 0.80) | argmax acc 0.9610 [358s]
epoch 4 | train loss 0.0433 | val score 0.9390 (acc 0.9480, oos recall 0.9300, tau 0.82) | argmax acc 0.9653 [484s]
epoch 5 | train loss 0.0259 | val score 0.9397 (acc 0.9293, oos recall 0.9500, tau 0.94) | argmax acc 0.9663 [609s]

BEST epoch 3 | tau 0.80 | in-scope acc 0.9370 | oos recall 0.9500 | score 0.9435 | argmax acc 0.9610


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

  ...n4cq2ii/model.safetensors:   0%|          | 14.2kB /  438MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

uploaded: https://huggingface.co/goncayldz/clinc150-bert-teacher


In [6]:
import gc, json, time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from huggingface_hub import upload_file

# Önceki eğitimden kalan GPU belleğini boşalt
for n in ["model", "opt", "loader", "sched"]:
    globals().pop(n, None)
gc.collect()
torch.cuda.empty_cache()

TEACHER_REPO = "goncayldz/clinc150-bert-teacher"
STUDENT_REPO = "goncayldz/clinc150-distilbert-student"
S_MODEL = "distilbert-base-uncased"
S_EPOCHS, S_LR, T, ALPHA = 6, 5e-5, 2.0, 0.5

# --- 1) Tokenizer uyumu: student ile teacher AYNI input_ids'i üretmeli ---
tok_s = AutoTokenizer.from_pretrained(S_MODEL)
sample = [normalize(t) for t in ds["train"]["text"][:200]]
assert tok(sample)["input_ids"] == tok_s(sample)["input_ids"], "tokenizer farkli!"
print("tokenizer uyumlu: ayni input_ids")

# --- 2) Teacher'ı HUB'dan indir, logit'leri bir kez hesapla ---
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_REPO).to(dev).eval()

@torch.no_grad()
def get_logits(m, ids, mask, bs=256):
    m.eval()
    out = []
    for i in range(0, len(ids), bs):
        out.append(m(input_ids=ids[i:i + bs].to(dev),
                     attention_mask=mask[i:i + bs].to(dev)).logits.float().cpu())
    return torch.cat(out)

tr_t = get_logits(teacher, tr_ids, tr_mask)
va_t = get_logits(teacher, va_ids, va_mask)
p_va = torch.softmax(va_t, dim=-1).numpy()
print("teacher (Hub'dan) val argmax acc:", round(in_scope_accuracy(yva, p_va.argmax(axis=1), OOS), 4),
      "| params (M):", round(sum(x.numel() for x in teacher.parameters()) / 1e6, 1))
print("teacher TRAIN argmax acc:", round(float((tr_t.argmax(1) == tr_y).float().mean()), 4))
del teacher
gc.collect()
torch.cuda.empty_cache()

# --- 3) Distillation loss ---
def kd_loss(s_logits, t_logits, y, alpha, T):
    ce = F.cross_entropy(s_logits, y)
    kd = F.kl_div(F.log_softmax(s_logits / T, dim=-1),
                  F.softmax(t_logits / T, dim=-1),
                  reduction="batchmean") * (T * T)
    return alpha * ce + (1 - alpha) * kd

def train_student(alpha, tag):
    torch.manual_seed(0)
    stu = AutoModelForSequenceClassification.from_pretrained(
        S_MODEL, num_labels=len(names),
        id2label={i: n for i, n in enumerate(names)},
        label2id={n: i for i, n in enumerate(names)},
    ).to(dev)
    print(f"\n=== {tag} | alpha={alpha} | params (M): {sum(x.numel() for x in stu.parameters()) / 1e6:.1f}")

    ld = DataLoader(TensorDataset(tr_ids, tr_mask, tr_y, tr_t), batch_size=BATCH, shuffle=True)
    o = torch.optim.AdamW(stu.parameters(), lr=S_LR, weight_decay=0.01)
    steps = S_EPOCHS * len(ld)
    warm = int(0.1 * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(
        o, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / (steps - warm))))

    best_s = {"score": -1.0}
    t0 = time.time()
    for epoch in range(1, S_EPOCHS + 1):
        stu.train()
        total = 0.0
        for ids, mask, y, tl in ld:
            ids, mask, y, tl = ids.to(dev), mask.to(dev), y.to(dev), tl.to(dev)
            o.zero_grad()
            loss = kd_loss(stu(input_ids=ids, attention_mask=mask).logits, tl, y, alpha, T)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(stu.parameters(), 1.0)
            o.step()
            sch.step()
            total += loss.item() * len(y)

        probs = get_probs(stu, va_ids, va_mask)
        score, tau, acc, rec = tune_tau(probs, yva, OOS)
        am = in_scope_accuracy(yva, probs.argmax(axis=1), OOS)
        print(f"epoch {epoch} | loss {total / len(tr_y):.4f} | val score {score:.4f} "
              f"(acc {acc:.4f}, oos recall {rec:.4f}, tau {tau:.2f}) | argmax acc {am:.4f} "
              f"[{time.time() - t0:.0f}s]")
        if score > best_s["score"]:
            best_s = {"score": score, "tau": tau, "acc": acc, "rec": rec, "epoch": epoch,
                      "argmax_acc": am,
                      "state": {k: v.detach().cpu().clone() for k, v in stu.state_dict().items()}}
    print(f"BEST {tag}: epoch {best_s['epoch']} | tau {best_s['tau']:.2f} | acc {best_s['acc']:.4f} | "
          f"oos recall {best_s['rec']:.4f} | score {best_s['score']:.4f} | argmax acc {best_s['argmax_acc']:.4f}")
    return best_s, stu

# --- 4) İki student: distillation'sız ve distillation'lı ---
res_plain, stu_plain = train_student(1.0, "PLAIN (no KD)")
del stu_plain
gc.collect()
torch.cuda.empty_cache()
res_kd, stu_kd = train_student(ALPHA, "KD (distilled)")

# --- 5) KD student'ı Hub'a yükle ---
stu_kd.load_state_dict(res_kd["state"])
stu_kd.push_to_hub(STUDENT_REPO)
tok_s.push_to_hub(STUDENT_REPO)
json.dump({"tau": res_kd["tau"], "epoch": res_kd["epoch"], "alpha": ALPHA, "T": T},
          open("tau_student.json", "w"))
upload_file(path_or_fileobj="tau_student.json", path_in_repo="tau.json", repo_id=STUDENT_REPO)
print("uploaded: https://huggingface.co/" + STUDENT_REPO)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer uyumlu: ayni input_ids


config.json:   0%|          | 0.00/8.40k [00:00<?, ?B/s]

... (15 outputs left out)

  ...wl48yho/model.safetensors:   0%|          |  574kB /  268MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

uploaded: https://huggingface.co/goncayldz/clinc150-distilbert-student
